# Parse messages\nSplit text records into classified, protocol-neutral message rows.

In [ ]:
project_root = "."
source = "data/capture"
fix_dictionary = None
protocols = None
pattern = "*.log*"
header = None
recursive = True
spill = False
timezone = None
include_regexes = []
exclude_regexes = []
include_msgtypes = []
exclude_msgtypes = []
technical_plugins = ["jolokia"]
plugin_keys = {}
null_values = ["", "null", "<null>", "n/a", "none"]
start = None
end = None
duration_ns = None
catalog = {"name": "rekep", "properties": {}}
table_properties = {"history.expire.max-snapshot-age-ms": "604800000"}
branch = "root"
target = "logs.messages"
merge_by = True
batch_row_size = 65_536
batch_byte_size = 67_108_864
max_row_byte_size = 67_108_864
commit_batch_num = 8
commit_row_size = None
limit = None
log_level = "INFO"

In [ ]:
from rekep import ArrowPath
from rekep.fix import FixCodec, FixRegistry
from rekep.fix.rules import Rules
from rekep.iceberg import IcebergCatalog
from rekep.logs import Stage, configure
from rekep.text import TextFiles
from rekep.times import unix_of
from rekep.urls import Url

configure(log_level)
if isinstance(commit_batch_num, bool) or not isinstance(commit_batch_num, int):
    raise TypeError("commit_batch_num must be an integer")
if commit_batch_num <= 0:
    raise ValueError("commit_batch_num must be positive")
if commit_row_size is not None and (
    isinstance(commit_row_size, bool) or not isinstance(commit_row_size, int)
):
    raise TypeError("commit_row_size must be an integer or null")
if commit_row_size is not None and commit_row_size <= 0:
    raise ValueError("commit_row_size must be positive")
lower, upper = unix_of(start), unix_of(end, upper=True)
registry = (
    FixRegistry()
    if fix_dictionary is None
    else FixRegistry(cache_dir=Url.from_string(str(fix_dictionary)).resolve(project_root))
)


protocol_rules = Rules.into_default() if protocols is None else Rules.from_dict(protocols)
codec = FixCodec(
    registry=registry,
    rules=protocol_rules,
    null_values=frozenset(null_values),
)
declared = {
    "timezone": timezone,
    "protocol_codec": codec,
    "msg_type_event_types": registry.msg_type_event_types(),
    "plugin_keys": plugin_keys,
    "null_values": null_values,
    "spill": spill,
    **({} if header is None else {"header_pattern": header}),
}
location = ArrowPath(str(source)).resolve(project_root)
rows = TextFiles.from_folder(
    location,
    start=start,
    end=end,
    pattern=pattern,
    recursive=recursive,
    **declared,
)
if not rows.exists:
    raise FileNotFoundError(location)
field = rows.into_struct_field()
stage = Stage(
    "parse_messages",
    sources={"capture": str(location)},
    targets={"messages": target},
    window=(lower, upper),
)

In [ ]:
store = IcebergCatalog.from_dict(catalog)
messages = store.dataset(
    target,
    field=field,
    table_properties=dict(table_properties),
    branch=branch,
    commit_batch_num=commit_batch_num,
    commit_row_size=commit_row_size,
)

counts = {"read": 0}


def _batches():
    reader = rows.read_arrow_reader(
        batch_row_size=batch_row_size,
        batch_byte_size=batch_byte_size,
        max_row_byte_size=max_row_byte_size,
        include_regexes=include_regexes,
        exclude_regexes=exclude_regexes,
        include_msgtypes=include_msgtypes,
        exclude_msgtypes=exclude_msgtypes,
        technical_plugins=technical_plugins,
        start_unix=lower,
        end_unix=upper,
        duration_ns=duration_ns,
    )
    try:
        for batch in reader:
            if limit is not None and counts["read"] + batch.num_rows > limit:
                batch = batch.slice(0, max(0, limit - counts["read"]))
            if batch.num_rows:
                counts["read"] += batch.num_rows
                yield batch
            if limit is not None and counts["read"] >= limit:
                break
    finally:
        reader.close()


written = messages.append_arrow_reader(
    _batches(),
    field,
    merge_by=merge_by,
    commit_row_size=commit_row_size,
    commit_batch_num=commit_batch_num,
)
result = stage.finished(read=counts["read"], written=written)
result